# Step 3: Cut the recording into daily files

This notebook belongs to the Geo-OBS manual, Session 6 ("Cutting Long-Term SEG-Y Files into Daily Files").

After step 2 every station has one file per component and per chunk folder (1.7 GB/file per 10 days). Most analyses look at a few hours or one day, and reading 1.7 GB to plot one hour is slow. This notebook joins the chunks of a station into one continuous recording and cuts it into files of one day (about 170 MB each).

**Input:** the `x`/`y`/`z`/`h` files from `2_fix_header_and_xyz.ipynb`:

```
<DATA_FOLDER>/<station>/1stChunk/<station>_1stChunk_x.segy, ..._y.segy, ..._z.segy, ..._h.segy
<DATA_FOLDER>/<station>/2ndChunk/...
```

**Output:** one file per day and component in a new folder next to the chunk folders:

```
<DATA_FOLDER>/<station>/DailyChunk/<station>_z_chunk0000_20260707T160000.segy
                                   <station>_z_chunk0001_20260708T160000.segy   ...
```

The number after `chunk` counts the days from `START_LOCAL`, the date is the start of the file in **UTC**.

**How the time is kept right:** Every input trace is placed at the time written in its own trace header, not just appended. If a minute is missing (for example between two EDR exports), the output contains zeros there and the notebook prints a warning. If two chunks overlap, the doubled minutes land on the same place. This matters: in 2026 a missing minute that was simply closed up made all later data of a station 59 s too early.

**Memory and time:** only one output file (one day) is in memory at a time, so a normal laptop is enough. One station with four components and 27 days takes about 15 to 30 minutes on an external hard drive.

You only have to change the cell **Settings**.

## Settings

In [ ]:
from pathlib import Path

DATA_FOLDER = Path("/ur/path/to/data")   # folder that contains one folder per station
STATIONS = ["0498"]                               # e.g. ["0498", "0578"] for several stations
COMPONENTS = ["x", "y", "z", "h"]

# Start of the first output file, in local time. Use the deployment time rounded to the full hour,
# plus settling time if you want to leave out the first hour on the seafloor.
START_LOCAL = "2026-07-07 18:00"
END_LOCAL = None                                  # None = until the end of the recording, or e.g. "2026-08-03 12:00"
UTC_OFFSET_HOURS = 2                              # the Trilobit records UTC; Svalbard summer time is UTC+2

HOURS_PER_FILE = 24                               # 24 = daily files; 1 = hourly files
OUTPUT_FOLDER = "DailyChunk"                      # name of the new folder in every station folder

## Helper functions

In [ ]:
import re
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

HEADER_BYTES = 3600


def find_chunk_folders(station_folder):
    return sorted(p for p in station_folder.iterdir()
                  if p.is_dir() and re.match(r"^\d+(st|nd|rd|th)Chunk$", p.name))


def trace_start_time(trace_header):
    year, day, hour, minute, second = struct.unpack(">5H", trace_header[156:166])
    millisecond, microsecond = struct.unpack(">2H", trace_header[168:172])
    return datetime(year, 1, 1) + timedelta(days=day - 1, hours=hour, minutes=minute, seconds=second,
                                            milliseconds=millisecond, microseconds=microsecond)


def set_trace_start_time(trace_header, time):
    trace_header[156:166] = struct.pack(">5H", time.year, time.timetuple().tm_yday, time.hour, time.minute, time.second)
    trace_header[166:168] = struct.pack(">H", 2)                                   # time basis code 2 = UTC
    trace_header[168:172] = struct.pack(">2H", time.microsecond // 1000, time.microsecond % 1000)


def file_info(path):
    """Reads only the first and the last trace header."""
    with open(path, "rb") as f:
        f.seek(HEADER_BYTES)
        first = f.read(240)
        n_samples, dt_us = struct.unpack(">2H", first[114:118])
        trace_type = np.dtype([("header", "V240"), ("data", ">f4", n_samples)])
        n_traces = (path.stat().st_size - HEADER_BYTES) // trace_type.itemsize
        f.seek(HEADER_BYTES + (n_traces - 1) * trace_type.itemsize)
        last = f.read(240)
    dt = dt_us / 1e6
    return dict(path=path, n_samples=n_samples, dt=dt, n_traces=n_traces, trace_type=trace_type,
                trace_seconds=n_samples * dt, start=trace_start_time(first),
                end=trace_start_time(last) + timedelta(seconds=n_samples * dt))

## Cut one component of one station

`cut_component` goes through the output files one by one. For each output file it

1. makes an empty time series of the right length (zeros),
2. reads only the input traces that fall into this time window (it jumps directly to them instead of reading the whole file),
3. puts every trace at the position given by its trace header time,
4. writes the file with 60 s traces and correct trace times, and frees the memory.

It returns one row per output file with the noise level (standard deviation, so the constant offset of the geophones does not count) and the maximum amplitude, for the overview at the end.

In [ ]:
def cut_component(station, component, start_utc, end_utc):
    inputs = sorted((file_info(folder / f"{station}_{folder.name}_{component}.segy")
                     for folder in find_chunk_folders(DATA_FOLDER / station)
                     if (folder / f"{station}_{folder.name}_{component}.segy").exists()),
                    key=lambda info: info["start"])
    if not inputs:
        print(f"{station} {component}: no input files found")
        return []

    first = inputs[0]
    dt, n_samples, trace_seconds = first["dt"], first["n_samples"], first["trace_seconds"]
    with open(first["path"], "rb") as f:
        file_header = bytearray(f.read(HEADER_BYTES + 240))
    trace_header_template = file_header[HEADER_BYTES:]
    file_header = file_header[:HEADER_BYTES]

    start = max(start_utc, first["start"])
    end = inputs[-1]["end"] if end_utc is None else min(end_utc, inputs[-1]["end"])
    output_folder = DATA_FOLDER / station / OUTPUT_FOLDER
    output_folder.mkdir(exist_ok=True)

    rows = []
    index = 0
    window_start = start
    while window_start < end:
        window_end = min(window_start + timedelta(hours=HOURS_PER_FILE), end)
        n = round((window_end - window_start).total_seconds() / dt)
        data = np.zeros(n, dtype=np.float32)
        filled = np.zeros(n, dtype=bool)

        for info in inputs:
            if info["end"] <= window_start or info["start"] >= window_end:
                continue                                                            # file not in this window
            # traces of this file that can fall into the window (one trace margin on both sides)
            i_first = max(0, int((window_start - info["start"]).total_seconds() // info["trace_seconds"]) - 1)
            i_last = min(info["n_traces"] - 1, int((window_end - info["start"]).total_seconds() // info["trace_seconds"]) + 1)
            traces = np.fromfile(info["path"], dtype=info["trace_type"], count=i_last - i_first + 1,
                                 offset=HEADER_BYTES + i_first * info["trace_type"].itemsize)
            for trace in traces:
                position = round((trace_start_time(trace["header"].tobytes()) - window_start).total_seconds() / dt)
                a, b = max(0, position), min(n, position + info["n_samples"])
                if b > a:
                    data[a:b] = trace["data"][a - position:b - position]
                    filled[a:b] = True

        # write: 60 s traces, header of the first input trace with new trace number and start time
        n_traces = int(np.ceil(n / n_samples))
        output = np.zeros(n_traces, dtype=first["trace_type"])
        padded = np.zeros(n_traces * n_samples, dtype=np.float32)        # last trace filled up with zeros
        padded[:n] = data
        output["data"] = padded.reshape(n_traces, n_samples)
        for i in range(n_traces):
            header = bytearray(trace_header_template)
            header[0:8] = struct.pack(">2I", i + 1, i + 1)                         # trace number
            set_trace_start_time(header, window_start + timedelta(seconds=i * trace_seconds))
            output["header"][i] = bytes(header)

        path = output_folder / f"{station}_{component}_chunk{index:04d}_{window_start:%Y%m%dT%H%M%S}.segy"
        with open(path, "wb") as f:
            f.write(file_header)
            output.tofile(f)

        missing = (n - int(filled.sum())) * dt
        print(f"  {path.name}: {n_traces} traces" + (f"   WARNING: {missing:.0f} s without data (zeros)" if missing else ""))
        rows.append(dict(station=station, component=component, start_utc=window_start, file=path.name,
                         traces=n_traces, missing_seconds=missing,
                         noise=float(data[filled].astype(np.float64).std()) if filled.any() else 0.0,
                         max_abs=float(np.abs(data).max())))
        index += 1
        window_start = window_end
    return rows

## Run

In [ ]:
start_utc = datetime.strptime(START_LOCAL, "%Y-%m-%d %H:%M") - timedelta(hours=UTC_OFFSET_HOURS)
end_utc = None if END_LOCAL is None else datetime.strptime(END_LOCAL, "%Y-%m-%d %H:%M") - timedelta(hours=UTC_OFFSET_HOURS)
print(f"Start {start_utc} UTC, end {'end of the recording' if end_utc is None else f'{end_utc} UTC'}, "
      f"{HOURS_PER_FILE} h per file")

rows = []
for station in STATIONS:
    for component in COMPONENTS:
        print(f"{station} {component}:")
        rows += cut_component(station, component, start_utc, end_utc)
overview = pd.DataFrame(rows)

## Overview

One point per output file. The noise level shows quiet and noisy days at a glance; the maximum shows days with large events. Use it to pick days for `4_qc_daily_files.ipynb`. A file with missing seconds is listed below the plot.

In [ ]:
for station, table in overview.groupby("station"):
    fig, (ax_rms, ax_max) = plt.subplots(1, 2, figsize=(14, 4))
    for component, part in table.groupby("component"):
        ax_rms.plot(part["start_utc"], part["noise"], marker="o", label=component)
        ax_max.plot(part["start_utc"], part["max_abs"], marker="o", label=component)
    ax_rms.set_title(f"{station}: noise level (standard deviation) per file")
    ax_max.set_title(f"{station}: maximum amplitude per file")
    for ax in (ax_rms, ax_max):
        ax.set_yscale("log")
        ax.set_xlabel("Start of the file (UTC)")
        ax.legend()
        ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.show()

gaps = overview[overview["missing_seconds"] > 0]
print("No missing data." if gaps.empty else gaps[["station", "component", "file", "missing_seconds"]].to_string())